# 05 · Comparison: base vs SFT vs PPO vs GRPO

No training here. We load the golden-set results saved by notebooks 02–04 and compare:
- **Scores** per category (mcq, red_flag, unsafe) and overall.
- **Wins and losses**: which items each method fixed or broke compared with SFT.
- **Examples**: the same question answered by every model.

Runs on CPU in seconds (only reads JSON files). Needs `results/02_golden_scores.json`, `03_ppo_scores.json`, `04_grpo_scores.json`.

## 1. Load results

In [ ]:
import sys, json
import pandas as pd
import matplotlib.pyplot as plt
sys.path.insert(0, "src")
from medrl.config import RESULTS_DIR
from medrl.evaluate import load_golden

runs = json.load(open(RESULTS_DIR / "02_golden_scores.json"))   # {"base": {...}, "sft": {...}}
for name, f in [("ppo", "03_ppo_scores.json"), ("grpo", "04_grpo_scores.json")]:
    p = RESULTS_DIR / f
    if p.exists():
        runs[name] = json.load(open(p))
    else:
        print("missing", f)
print(list(runs))

## 2. Score table and chart

In [ ]:
scores = pd.DataFrame({m: r["scores"] for m, r in runs.items()}).T
scores = scores[["mcq", "red_flag", "unsafe", "overall"]]
display(scores.round(3))
scores.plot.bar(figsize=(9, 4), rot=0, ylim=(0, 1), title="Golden set accuracy (200 items)")
plt.ylabel("accuracy"); plt.tight_layout()
plt.savefig(RESULTS_DIR / "05_comparison.png", dpi=150)
scores.to_csv(RESULTS_DIR / "05_comparison.csv")

## 3. Is the difference real?

With 150 MCQ items, one item is 0.7 points. A rough 95% margin for an accuracy *p* on *n* items is ±1.96·√(p(1−p)/n). Differences smaller than that may be noise.

In [ ]:
n = {"mcq": 150, "red_flag": 30, "unsafe": 20, "overall": 200}
margin = scores.apply(lambda col: 1.96 * (col * (1 - col) / n[col.name]) ** 0.5)
display(margin.round(3))

## 4. Fixed and broken items vs SFT

In [ ]:
per_item = pd.DataFrame({m: {r["id"]: r["score"] for r in run["rows"]} for m, run in runs.items()})
for m in [c for c in per_item.columns if c not in ("base", "sft")]:
    fixed = ((per_item["sft"] == 0) & (per_item[m] == 1)).sum()
    broke = ((per_item["sft"] == 1) & (per_item[m] == 0)).sum()
    print(f"{m}: fixed {fixed}, broke {broke}, net {fixed - broke:+d}")

## 5. Same question, every model

In [ ]:
golden = {g["id"]: g for g in load_golden()}
answers = {m: {r["id"]: r["answer"] for r in run["rows"]} for m, run in runs.items()}
for t in ["red_flag", "unsafe", "mcq"]:
    gid = next(g for g in golden if golden[g]["type"] == t)
    print("=" * 80); print(t.upper(), ":", golden[gid]["prompt"][:200])
    for m in answers:
        print(f"--- {m} (score {per_item.loc[gid, m]:.0f}):", answers[m][gid][:300].replace(chr(10), " "))

## 6. Takeaways (fill in after running)

- **SFT** teaches the answer format and tone; most of the jump from base usually comes from here.
- **PPO** optimises a *learned* reward model: watch for answers that please the reward model but not the rules.
- **GRPO** optimises the *rules* directly: strong on red_flag / unsafe, but check that it doesn't just spam keywords.
- Use the margins in section 3 before claiming one method is better.